# 🤗 HuggingFace Open-Source Q&A — Mistral 7B

מחברת זו מורידה את **Mistral-7B-Instruct** מ-HuggingFace ומאפשרת לשאול שאלות ולקבל תשובות.

| תכונה | פרטים |
|--------|--------|
| מודל | `mistralai/Mistral-7B-Instruct-v0.2` |
| פרמטרים | 7B |
| Quantization | 4-bit (bitsandbytes) — ~4GB VRAM |
| רישיון | Apache 2.0 |
| GPU מינימלי | T4 (Colab חינמי) |

> **הפעל GPU לפני הרצה:** Runtime → Change runtime type → T4 GPU

## 1. התקנת ספריות

In [ ]:
!pip install transformers torch accelerate bitsandbytes sentencepiece -q

## 2. בחירת מודל

| מודל | פרמטרים | VRAM (4-bit) | הערות |
|------|---------|--------------|-------|
| `mistralai/Mistral-7B-Instruct-v0.2` | 7B | ~4GB | ברירת מחדל, מצוין |
| `mistralai/Mixtral-8x7B-Instruct-v0.1` | 47B MoE | ~24GB | דורש Colab Pro+ |
| `microsoft/Phi-3-mini-4k-instruct` | 3.8B | ~2.5GB | קל יותר, T4 בנוחות |
| `google/gemma-2b-it` | 2B | ~1.5GB | המהיר ביותר |

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
# MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
# MODEL_NAME = "google/gemma-2b-it"
# MODEL_NAME = "mistralai/Mixtral-8x7B-Instruct-v0.1"  # דורש Colab Pro+

## 3. טעינת המודל עם 4-bit Quantization

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "GPU not found! Enable GPU: Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Loading model: {MODEL_NAME} (4-bit quantization)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"
✅ Model loaded! VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 4. פונקציית שאלה-תשובה

In [ ]:
def ask(question: str, max_new_tokens: int = 512, temperature: float = 0.7) -> str:
    messages = [{"role": "user", "content": question}]
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = f"[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][input_len:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 5. שאלות לדוגמה

In [ ]:
example_questions = [
    "What is the capital of France?",
    "Explain quantum entanglement in simple terms.",
    "What are the main differences between Python and JavaScript?",
    "מה הבירה של ישראל?",
]

for q in example_questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 60)

## 6. ממשק אינטראקטיבי

הרץ את התא הבא ושאל שאלות חופשיות. כתוב `exit` כדי לסיים.

In [ ]:
print("🤖 Mistral Q&A Bot ready! Type 'exit' to quit.
")
while True:
    question = input("Your question: ").strip()
    if not question:
        continue
    if question.lower() in ("exit", "quit", "q", "יציאה"):
        print("Goodbye!")
        break
    print(f"Answer: {ask(question)}
")